# 02 — Vector Search over Internal Research

Makes the internal `research_documents` table semantically searchable with
**Mosaic AI Vector Search**. The agent (notebook `05`) uses the resulting index
through `VectorSearchRetrieverTool` to answer questions like *"What is our internal
thesis on NVIDIA?"* from proprietary content that never leaves your workspace.

We use a **Delta Sync index** with **Databricks-managed embeddings** — you point it
at the Delta table and an embedding model endpoint, and Databricks keeps the index
in sync as the table changes (Change Data Feed was enabled on the table in `01`).

In [0]:
%pip install -U databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
CATALOG = "bigdata_demo"
SCHEMA = "financial_intelligence"

VS_ENDPOINT = "bigdata_demo_vs"                # Vector Search endpoint (compute)
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.research_documents"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.research_docs_index"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"  # managed foundation-model embeddings

print(f"Endpoint: {VS_ENDPOINT}")
print(f"Source:   {SOURCE_TABLE}")
print(f"Index:    {INDEX_NAME}")

## 1. Create (or reuse) a Vector Search endpoint

An endpoint is the serving compute for one or more indexes. Creation takes a few
minutes the first time; the cell waits until it is online.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

existing = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if VS_ENDPOINT not in existing:
    vsc.create_endpoint_and_wait(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"Created endpoint {VS_ENDPOINT}")
else:
    print(f"Endpoint {VS_ENDPOINT} already exists")

## 2. Create the Delta Sync index with managed embeddings

> **⏳ This cell can take a while — expect ~5–20 minutes, and potentially longer
> depending on your selected compute/serverless size and whether the embedding
> endpoint has to cold-start.** The `_and_wait` call intentionally blocks until the
> index is provisioned, embedded, and fully synced online. The tiny row count does
> **not** make it fast — provisioning and the embedding endpoint's cold-start dominate.
> It is normal for this to sit "running" for several minutes with no output. To confirm
> it is progressing (not stuck), watch the index status under **Catalog → bigdata_demo
> → financial_intelligence → research_docs_index** move `PROVISIONING → ONLINE`, or
> check the endpoint under **Compute → Vector Search**. Only investigate if it exceeds
> ~20 min with no status change or throws an embedding-endpoint error.

In [0]:
try:
    index = vsc.create_delta_sync_index_and_wait(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="doc_id",
        embedding_source_column="content",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
    )
    print(f"Created index {INDEX_NAME}")
except Exception as e:
    # Index already exists — just sync it
    print(f"Index may already exist ({e}); triggering a sync instead.")
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
    index.sync()

## 3. Smoke test — semantic search

In [0]:
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
results = index.similarity_search(
    query_text="What are the key risks in the technology sector?",
    columns=["doc_id", "title", "company", "doc_type"],
    num_results=3,
)
for row in results["result"]["data_array"]:
    print(row)

## Done

The internal research index is live. Continue with **`03_bigdata_mcp_setup`** to
connect the external Bigdata.com MCP server.